# (노트) 토픽모델, R예제
> 작성완료

- toc:true
- branch: master
- badges: true
- comments: true
- author: 신록예찬
- categories: [통계계산, 딥러닝, R]

### About this doc

`-` 전북대 강의노트

### Toy Example

In [106]:
library(tidyverse)
documents = rbind(c('손흥민','골'),
                  c('골','확률'),
                  c('확률','데이터과학'))

In [107]:
rownames(documents)<-c('doc1','doc2','doc3')
colnames(documents)<-c('word1','word2')
documents

,word1,word2
doc1,손흥민,골
doc2,골,확률
doc3,확률,데이터과학


In [108]:
document_topics=rbind(c(1,0),
                      c(0,1),
                      c(1,0)) ## 랜덤으로 초기화 
rownames(document_topics)<-c('doc1','doc2','doc3')
colnames(document_topics)<-c('topic1','topic2')

In [109]:
document_topics

,topic1,topic2
doc1,1,0
doc2,0,1
doc3,1,0


문서당 토픽비율 
- 문서1: [손흥민, 골] = [1,0]
- 문서2: [골, 확률] = [0,1]
- 문서3: [확률, 데이터과학] = [1,0]

토픽별로 자주 등장하는 단어 
- 토픽0: 골, 골, 데이터과학 ---> 토픽이름은 축구??.. 
- 토픽1: 손흥민, 확률, 확률 ---> 토픽이름은 확률?? 

목표: document_topics의 값을 그럴듯하게 바꾸자. 예를들면 아래처럼. 
```r
document_topics=rbind(doc1=c(0,0),
                      doc2=c(0,1),
                      doc3=c(1,1)) ## 랜덤으로 초기화 
```
그러니까 $3 \times 2$ 차원 랜덤변수를 뽑는문제임. 

--- 

#### 문서1 : [`손흥민`, 골]

`-` 랜덤 초기화로 인한 현재상황은 아래와 같다. 

문서당 토픽비율 
- 문서1: [손흥민, 골] = [1,0]
- 문서2: [골, 확률] = [0,1]
- 문서3: [확률, 데이터과학] = [1,0]

토픽별로 자주 등장하는 단어 
- 토픽0: 골, 골, 데이터과학
- 토픽1: 손흥민, 확률, 확률 

`–` 위에는 임의의 초기값이 들어가있다. 우리는 지금 문서1에서 손흥민에 해당하는 토픽으로 `토픽=0`이 그럴듯한지 `토픽=1`이 그럴듯한지 다시 생각하고 싶다. 

`-` 깁스샘플링: 현재 관심있는 문서1의 첫단어 손흥민에 대한 토픽분류를 제외하고 나머지는 모두 올바른 값이라고 가정하자. 

문서당 토픽비율 
- 문서1: [손흥민, 골] = [1,0] ==> [?,0] 
- 문서2: [골, 확률] = [0,1] ==> [0,1]
- 문서3: [확률, 데이터과학] = [1,0] ==> [1,0]

토픽별로 자주 등장하는 단어 
- 토픽0: 골, 골, 데이터과학 ==> 골, 골, 데이터과학
- 토픽1: 손흥민, 확률, 확률 ==> ?, 확률, 확률

`-` 손흥민이라는 단어가 뽑힐 경우는 아래의 두 가지가 있다. 
 - 토픽0에서 뽑혔을 경우
 - 토픽1에서 뽑혔을 경우

`-` 토픽0에서 뽑혔을 경우는 아래와 같이 계산가능하다 
 - (문서1에 토픽0이 포함되어 있는 비율) $\times$ (토픽0에서 손흥민이라는 단어를 뽑을 확률) = 1 $\times$ 0.001 
 - p(토픽0|문서1) $\times$ p(손흥민|토픽0) = 1 $\times$ 0.001

`–` 토픽1에서 뽑혔을경우는 아래와 같이 계산가능하다 
 - p(토픽1|문서1) $\times$ p(손흥민|토픽1) = 0.001 $\times$ 0.001 

`–` 손흥민은 토픽0에서 뽑혔다고 보는것이 타당함. (왜? 현재단어 손흥민은 문서1에 있고, 문서1은 토픽0이 대부분이니까!) 

`–` 아래와 같이 업데이트 한다. 

문서당 토픽비율 
- 문서1: [손흥민, 골] = [0,0] 
- 문서2: [골, 확률] = [0,1] 
- 문서3: [확률, 데이터과학] = [1,0] 

토픽별로 자주 등장하는 단어 
- 토픽0: 손흥민, 골, 골, 데이터과학
- 토픽1: 확률, 확률 

#### 문서1:[손흥민,`골`]

`-` 업데이트 전

문서당 토픽비율 
- 문서1: [손흥민, 골] = [0,0] ==> [0,?]
- 문서2: [골, 확률] = [0,1] ==> [0,1]
- 문서3: [확률, 데이터과학] = [1,0] ==> [1,0]

토픽별로 자주 등장하는 단어 
- 토픽0: 손흥민, 골, 골, 데이터과학 ==> 손흥민, ?, 골, 데이터과학 
- 토픽1: 확률, 확률 ==> 확률, 확률 

`-` 샘플링 
- p(토픽0|문서1) $\times$ p(골|토픽0) = 1 $\times$ 1/3
- p(토픽1|문서1) $\times$ p(골|토픽1) = 0.001 $\times$ 0.001 

결정: 토픽0 선택 (왜? 현재단어는 문서1에 있는데 문서1에는 일단 토픽 0이 대부분이다 `+` 현재 단어는 골인데 골은 토픽0에서 잘나오니까) 

`-` 업데이트 후 

문서당 토픽비율 
- 문서1: [손흥민, 골] = [0,0]
- 문서2: [골, 확률] = [0,1]
- 문서3: [확률, 데이터과학] = [1,0]

---

`-` 이런식으로 반복하면 결국 각 단어는 자신의 정체성을 아래와 같은 2가지 기준으로 판단하게 된다. 

- 나는 $d$-th document에 속해있다 $\to$ 그런데 이 document의 단어를 쭉보니까 토픽 $p$ 에 많다 $\to$ 나도 토픽 $p$ 인가? 
- 나와 똑같은 이름을 가진 단어들이 토픽 $p'$에 많다 $\to$ 나도 토픽 $p'$에서 왔을까? 

`-` 재미있는점은 각 단어의 선택이 다른 단어에도 영향을 준다는 것이다. (다들 나 빼고는 다 맞다고 가정하고 있으니까.. )

--- 

### 구현

`-` 구현해보자. 

In [110]:
doclen<-5
word<-c('손흥민','골','골','박지성','패스',
        '골','확률','패스','손흥민','골',
        '골','골','확률','패스','골',
        '골','박지성','통계','확률','골',
        '확률','통계','박지성','통계','골',
        '확률','확률','골','통계','AI',
        '확률','확률','통계','통계','AI',
        '확률','빅데이터','데이터과학','빅데이터','AI',
        '확률','데이터과학','AI','데이터과학','데이터과학')
doc<-c(rep(1,doclen),rep(2,doclen),rep(3,doclen),
       rep(4,doclen),rep(5,doclen),rep(6,doclen),
       rep(7,doclen),rep(8,doclen),rep(9,doclen))
topic<-c(2,1,1,2,2,
         1,2,1,2,2,
         1,2,1,2,2,
         1,2,1,2,2,
         1,2,1,2,2,
         1,2,2,2,2,
         1,2,1,1,2,
         1,2,1,1,2,
         2,1,1,2,2)
data<- tibble(word=word,doc=doc,topic=topic)
head(data)

word,doc,topic
<chr>,<dbl>,<dbl>
손흥민,1,2
골,1,1
골,1,1
박지성,1,2
패스,1,2
골,2,1


In [111]:
topic_doc<-function(data,k,d,alpha=0.1){
    count_ = dim(filter(data,doc==d,topic==k))[1] + alpha
    total_ = dim(filter(data,doc==d))[1]+ K*alpha
    count_ / total_ 
}

In [112]:
word_topic<-function(data,w,k,alpha=0.1){
    word_=filter(data,doc==d)[w,]$word
    count_ = dim(filter(data,topic==k,word==word_))[1] + alpha 
    total_ = dim(filter(data,topic==k))[1] + N*alpha 
    count_ / total_ 
}

In [113]:
topicprob<-function(data,d,w){
    rtn<-c()
    for(k in 1:K){
       rtn[k] = topic_doc(data,k,d) * word_topic(data,w,k)
    }
    rtn/sum(rtn)
}

In [114]:
diri<-function(weights){
    cum<- weights %>% cumsum() 
    U<-runif(1)
    1+sum(U>cum)
}

In [115]:
D<-max(data$doc)
K<-2
N<-length(unique(data$word))
for(iter in 1:50){
    i<-0
    for(d in 1:D){
        for(w in 1:doclen){
            i<-i+1
            weights_<-topicprob(data[-i,],d,w)
            data[i,]$topic<-diri(weights_)
        }
    }
}
data

word,doc,topic
<chr>,<dbl>,<dbl>
손흥민,1,2
골,1,2
골,1,2
박지성,1,2
패스,1,1
골,2,2
확률,2,2
패스,2,2
손흥민,2,2
